# Time Series — Air Passengers: EDA & Decomposition

**Business Question:** What are the patterns in air passenger demand, and is the series predictable?

**Dataset:** Monthly international airline passengers, 1949–1960 (144 observations)

---
**Sections:**
1. Load Data
2. Time Series Visualization
3. Stationarity Test (ADF)
4. Transformations
5. Decomposition
6. ACF & PACF

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from statsmodels.datasets import get_rdataset
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load built-in AirPassengers dataset
data = get_rdataset('AirPassengers').data
data.columns = ['time', 'passengers']

# Build proper datetime index
start = pd.Timestamp('1949-01-01')
data.index = pd.date_range(start=start, periods=len(data), freq='MS')
data = data[['passengers']]

print(f'Shape: {data.shape}')
print(f'Period: {data.index[0].strftime("%b %Y")} → {data.index[-1].strftime("%b %Y")}')
data.head(12)

## 2. Time Series Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full series
axes[0].plot(data.index, data['passengers'], color='#4C72B0', linewidth=1.5)
axes[0].set_title('Monthly International Airline Passengers (1949–1960)',
                  fontsize=13, fontweight='bold')
axes[0].set_ylabel('Passengers (thousands)')
axes[0].set_xlabel('')

# Annual average overlay
annual = data.resample('Y').mean()
axes[0].plot(annual.index, annual['passengers'], color='#DD8452',
             linewidth=2.5, linestyle='--', label='Annual average')
axes[0].legend()

# Monthly seasonality boxplot
data['month'] = data.index.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_data = [data[data['month'] == m]['passengers'].values for m in range(1, 13)]
axes[1].boxplot(monthly_data, labels=month_names, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.5),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Passenger Distribution by Month — Seasonal Pattern',
                  fontsize=13, fontweight='bold')
axes[1].set_ylabel('Passengers (thousands)')

data.drop(columns=['month'], inplace=True)
plt.tight_layout()
plt.show()

In [ ]:
# Yearly growth
annual_mean = data.resample('Y').mean()
annual_mean['year'] = annual_mean.index.year
annual_mean['growth_pct'] = annual_mean['passengers'].pct_change() * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(annual_mean['year'][1:], annual_mean['growth_pct'][1:], color='#55A868', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Year-over-Year Passenger Growth (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Growth (%)')
ax.set_xlabel('Year')
for i, (year, val) in enumerate(zip(annual_mean['year'][1:], annual_mean['growth_pct'][1:])):
    ax.text(year, val + 0.3, f'{val:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Stationarity Test (ADF)

ARIMA requires a **stationary** series (constant mean and variance over time). We use the Augmented Dickey-Fuller (ADF) test.

- **H₀:** The series has a unit root (non-stationary)
- **H₁:** The series is stationary
- If p-value < 0.05 → reject H₀ → series is stationary

In [ ]:
def run_adf_test(series, name='Series'):
    """Run ADF test and print a clean summary."""
    result = adfuller(series.dropna())
    print(f'ADF Test — {name}')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Conclusion    : {"STATIONARY ✓" if result[1] < 0.05 else "NON-STATIONARY ✗"}')
    print()

run_adf_test(data['passengers'], 'Original series')

## 4. Transformations

The original series is non-stationary (growing trend + increasing variance). We apply:
1. **Log transform** — stabilizes variance
2. **First difference** — removes trend
3. **Seasonal difference (lag 12)** — removes seasonality

In [ ]:
data['log'] = np.log(data['passengers'])
data['log_diff'] = data['log'].diff()
data['log_diff_seasonal'] = data['log_diff'].diff(12)

fig, axes = plt.subplots(4, 1, figsize=(14, 14))

series_to_plot = [
    ('passengers', 'Original Series', '#4C72B0'),
    ('log', 'Log Transform (stabilizes variance)', '#DD8452'),
    ('log_diff', 'Log + First Difference (removes trend)', '#55A868'),
    ('log_diff_seasonal', 'Log + First Diff + Seasonal Diff (removes seasonality)', '#C44E52')
]

for ax, (col, title, color) in zip(axes, series_to_plot):
    ax.plot(data.index, data[col], color=color, linewidth=1.2)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Transformation Steps Toward Stationarity', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ADF tests on each transformation
run_adf_test(data['log'], 'Log transform')
run_adf_test(data['log_diff'], 'Log + First difference')
run_adf_test(data['log_diff_seasonal'], 'Log + First diff + Seasonal diff')

## 5. Decomposition

We decompose the log-transformed series into **trend**, **seasonality**, and **residuals** using multiplicative decomposition.

In [ ]:
decomposition = seasonal_decompose(data['log'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
components = [
    (data['log'], 'Observed (log)', '#4C72B0'),
    (decomposition.trend, 'Trend', '#DD8452'),
    (decomposition.seasonal, 'Seasonality', '#55A868'),
    (decomposition.resid, 'Residuals', '#C44E52')
]

for ax, (series, title, color) in zip(axes, components):
    ax.plot(data.index, series, color=color, linewidth=1.2)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Value')
    if title == 'Residuals':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('Time Series Decomposition (Additive, Log Scale)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. ACF & PACF

**ACF (Autocorrelation Function)** and **PACF (Partial ACF)** help identify the ARIMA parameters:
- ACF → determines the MA(q) order
- PACF → determines the AR(p) order
- Spikes at lag 12 → confirm seasonal component → use SARIMA

In [ ]:
stationary_series = data['log_diff_seasonal'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(stationary_series, lags=40, ax=axes[0], color='#4C72B0')
axes[0].set_title('Autocorrelation Function (ACF) — Stationary Series',
                  fontsize=12, fontweight='bold')

plot_pacf(stationary_series, lags=40, ax=axes[1], color='#DD8452')
axes[1].set_title('Partial Autocorrelation Function (PACF) — Stationary Series',
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Reading the plots:')
print('  PACF cuts off after lag 1 → AR(p) = 1')
print('  ACF cuts off after lag 1  → MA(q) = 1')
print('  Spikes at lag 12          → Seasonal component (S=12)')
print('  → Suggested model: SARIMA(1,1,1)(1,1,1,12)')

In [ ]:
# Save processed series for modeling notebook
data.to_csv('air_passengers_processed.csv')
print('Saved: air_passengers_processed.csv')
print(f'Final shape: {data.shape}')
data.tail()